# 00 — Setup & Overview
## Machine Learning in SPM Tutorial
*RMS AFM & SPM Meeting 2026*

## Goals

By the end of this tutorial, you will be able to:

- Understand why machine learning is increasingly relevant to scanning probe microscopy
- Recognise the main data formats encountered in SPM (images, spectra, hyperspectral cubes)
- Apply denoising and dimensionality-reduction techniques to SPM images and spectra
- Use unsupervised clustering to segment SPM data into physically meaningful regions
- Appreciate how active learning and Bayesian optimisation can guide where to measure next
- Run end-to-end examples on synthetic SPM-like datasets that mirror real experimental challenges

In [ ]:
# Quick environment check — make sure all required packages are available
import numpy
import scipy
import matplotlib
import sklearn

print("numpy  :", numpy.__version__)
print("scipy  :", scipy.__version__)
print("matplotlib:", matplotlib.__version__)
print("sklearn:", sklearn.__version__)
print()
print("All imports OK")

In [ ]:
# Add the tutorial src/ directory to the Python path so we can import
# the helper modules provided with this tutorial.
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

## Why Machine Learning for SPM?

Scanning probe microscopy is one of the most information-rich characterisation techniques available to materials scientists. A single AFM or STM session can produce thousands of images, each carrying nanometre-scale detail about surface structure, mechanical properties, electrical behaviour, or chemical identity. That richness is also the challenge: extracting meaning from so much data, quickly and consistently, is genuinely hard to do by eye.

Machine learning offers a set of tools that are well matched to this challenge. At its core, ML is about finding structure in data — and SPM data is full of structure, even when it is buried under noise or spread across hundreds of channels. Denoising algorithms can recover sharp features from images acquired at high speed or low signal-to-noise. Clustering methods can segment a surface into regions that share similar properties without the researcher having to define the boundaries by hand. Regression models can learn the relationship between a quickly measured proxy (say, a single-frequency amplitude image) and a slower, richer measurement (a full force–distance curve), so that the slow measurement only needs to be made where it matters most.

Perhaps most excitingly, ML can begin to close the loop between measurement and decision-making. In a traditional SPM experiment the scientist decides where to look next based on intuition and experience. Bayesian optimisation and active learning algorithms can make the same kind of decision automatically, prioritising locations that are most likely to be informative and steering the microscope towards the most interesting parts of the sample. The result is experiments that extract more science per hour of instrument time — a compelling prospect when access to an SPM is expensive and competitive.

## The Data Problem in SPM

It helps to think about the three main shapes that SPM data comes in before we dive into the methods.

**Images** are the most familiar format: a regular 2-D grid of pixels, each holding a single number (height, phase, current, …). Think of it like a greyscale photograph of the surface, except that each pixel value is a physical measurement rather than a light intensity. A typical image might be 512 × 512 pixels — roughly 260,000 numbers — and you might collect dozens of channels simultaneously.

**Spectra** add a third axis. Instead of one number per pixel, you record an entire curve — a force–distance relationship, a tunnelling spectrum, a Kelvin probe frequency sweep. A single spectrum might have 512 points; a map of 64 × 64 spectra contains over two million numbers. This is sometimes called a **hyperspectral** or **4D** dataset, and it is where conventional manual analysis really starts to struggle. The analogy here is a cinema film of the surface: each frame is an image at a particular value of the spectral axis (voltage, frequency, force, …).

All three formats share the same practical headaches: **noise** from thermal fluctuations and electronic interference, **drift** as the tip and sample move relative to each other during acquisition, and **sheer volume** — more data than any researcher can inspect pixel by pixel. The methods covered in this tutorial address each of these headaches in turn, always using the physical context of SPM to guide which tools make sense.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.synthetic.generators import make_domain_image, make_hyperspectral_spm

plt.style.use('default')

# --- Generate synthetic data ---
# Domain image: smooth domains separated by sharp walls (like a ferroelectric surface)
domain_img = make_domain_image(size=128, n_domains=6, noise=0.05, seed=42)

# Grain image: Voronoi-style polycrystalline microstructure
grain_img = make_domain_image(size=128, n_domains=20, noise=0.08, seed=7)

# Hyperspectral cube: shape (n_x, n_y, n_spec)
hcube, freq_axis = make_hyperspectral_spm(nx=64, ny=64, n_freq=128, seed=42)

# --- Build a 2x2 figure ---
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
fig.suptitle('Synthetic SPM Data — Overview', fontsize=14, fontweight='bold')

# Top-left: domain image
ax = axes[0, 0]
im0 = ax.imshow(domain_img, cmap='RdBu_r', origin='lower')
ax.set_title('Domain Image\n(e.g. piezoresponse phase)')
ax.set_xlabel('x (pixels)')
ax.set_ylabel('y (pixels)')
plt.colorbar(im0, ax=ax, label='Signal (a.u.)')

# Top-right: grain image
ax = axes[0, 1]
im1 = ax.imshow(grain_img, cmap='viridis', origin='lower')
ax.set_title('Grain Image\n(e.g. contact stiffness)')
ax.set_xlabel('x (pixels)')
ax.set_ylabel('y (pixels)')
plt.colorbar(im1, ax=ax, label='Signal (a.u.)')

# Bottom-left: a few representative spectra from the hyperspectral cube
ax = axes[1, 0]
rng = np.random.default_rng(0)
sample_pixels = rng.integers(0, 64, size=(6, 2))
for px, py in sample_pixels:
    ax.plot(freq_axis, hcube[px, py, :], alpha=0.7, linewidth=1.2)
ax.set_title('Representative Spectra\n(6 random pixel locations)')
ax.set_xlabel('Frequency (a.u.)')
ax.set_ylabel('Response (a.u.)')
ax.grid(True, alpha=0.3)

# Bottom-right: one spectral slice of the hyperspectral cube
ax = axes[1, 1]
slice_idx = hcube.shape[2] // 2
im3 = ax.imshow(hcube[:, :, slice_idx], cmap='plasma', origin='lower')
ax.set_title(f'Hyperspectral Cube — Slice {slice_idx}\n(one frequency channel)')
ax.set_xlabel('x (pixels)')
ax.set_ylabel('y (pixels)')
plt.colorbar(im3, ax=ax, label='Response (a.u.)')

plt.tight_layout()
plt.show()

## Tutorial Modules

The tutorial is split into four core notebooks, each self-contained but building on concepts introduced here.

| Notebook | Topic | One-line description |
|---|---|---|
| `01_denoising.ipynb` | Denoising SPM Images | Compare classical filters with learned denoisers on noisy AFM images |
| `02_dimensionality_reduction.ipynb` | Dimensionality Reduction | Use PCA and UMAP to compress hyperspectral SPM cubes into interpretable maps |
| `03_clustering_segmentation.ipynb` | Clustering & Segmentation | Automatically segment surfaces into physically distinct regions with k-means and GMMs |
| `04_active_learning.ipynb` | Active Learning & Bayesian Optimisation | Let an algorithm decide where to measure next to find a target property efficiently |

## Setup Notes

**Running the notebooks.** All notebooks are designed to be run top-to-bottom in order, from cell 1 to the last cell. If you encounter an import error, re-run the `sys.path` cell above (the second code cell) and try again. Kernel restarts are occasionally needed after installing new packages — use *Kernel → Restart & Run All* in JupyterLab or classic Notebook.

**Synthetic data.** Every dataset used in this tutorial is generated on the fly from the `src/synthetic/generators` module. There are no large data files to download. The synthetic images and spectra are designed to mimic the statistical properties of real SPM data — noise characteristics, domain structures, spectral line shapes — while remaining fully reproducible via fixed random seeds. This means you can run the tutorial offline and on modest hardware.

**Package requirements.** The full list of required packages is in `requirements.txt` at the root of the repository. If you are running on your own machine and have not yet installed them, open a terminal in the repository root and run:

```bash
pip install -r requirements.txt
```

If you are using the pre-configured tutorial environment provided at the meeting, everything should already be installed and the import-check cell above will confirm this.